Visualisierungen der einzelnen Klassen

In [1]:
file_path = "../whatamidoing.geojson"

Anzahl Geschosse

In [2]:
import geopandas as gpd
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import json

# ── LOAD DATA FROM GEOJSON ────────────────────────────────────────────────────
gdf = gpd.read_file(
    "../whatamidoing.geojson"
)

print(gdf.columns.tolist())


# ── KEY FIX: fill NaN BEFORE value_counts ────────────────────────────────────
col = "gastw"   # change if the name differs in the GeoJSON
series = gdf[col].fillna("Keine Angabe").astype(str)

counts = series.value_counts().reset_index()
counts.columns = ["Geschosse", "Anzahl"]

def sort_key(val):
    try:
        return (0, float(val))
    except:
        return (1, val)

counts = counts.sort_values("Geschosse", key=lambda x: x.map(sort_key)).reset_index(drop=True)

# ── CHART ─────────────────────────────────────────────────────────────────────
fig = go.Figure(go.Bar(
    x=counts["Geschosse"],
    y=counts["Anzahl"],
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    title={
        "text": "Anzahl Geschosse – Verteilung CH (GeoJSON)"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: projected_buildings_enriched_ch.geojson | inkl. fehlende Werte</span>"
    }
)
fig.update_xaxes(title_text="Geschosse", type="category")
fig.update_yaxes(title_text="Anzahl Gebäude")

fig.write_image("anzahl_geschosse_geojson.png")
with open("anzahl_geschosse_geojson.png.meta.json", "w") as f:
    json.dump({
        "caption": "Verteilung Anzahl Geschosse (GeoJSON, inkl. fehlende Werte)",
        "description": "Bar chart of floor count distribution from projected_buildings_enriched_ch.geojson including missing values"
    }, f)

print(counts.to_string(index=False))


['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
   Geschosse  Anzahl
         1.0  314730
         2.0  868238
         3.0  589106
         4.0  184955
         5.0   71280
         6.0   30704
         7.0   15764
         8.0    7575
         9.0    3543
        10.0    1581
        11.0     755
        12.0     515
        13.0     382
        14.0     252
        15.0     200
        16.0     145
        17.0      73
        18.0 

Anzahl EGIDS

In [1]:
import geopandas as gpd
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file(
    "../whatamidoing.geojson"
)

print(gdf.columns.tolist())

# ── COUNT EGID COVERAGE ───────────────────────────────────────────────────────
# Normalize EGID column first (handles '', ' ', '0', etc.)
egid_raw = gdf["egid"].astype(str).str.strip()

has_egid = egid_raw.notna() & (egid_raw != "") & (egid_raw != "0")
total = len(gdf)
count_with    = has_egid.sum()
count_without = total - count_with
pct_with    = count_with / total * 100
pct_without = count_without / total * 100

print(f"Total features:    {total:,}")
print(f"Mit EGID:          {count_with:,}  ({pct_with:.1f}%)")
print(f"Ohne EGID:         {count_without:,} ({pct_without:.1f}%)")


# ── PIE CHART ─────────────────────────────────────────────────────────────────
labels = ["Mit EGID", "Ohne EGID"]
values = [count_with, count_without]

fig = go.Figure(go.Pie(
    labels=labels,
    values=values,
    textinfo="label+percent+value",
    texttemplate="%{label}<br>%{percent}<br>%{value:,}",
    hole=0.3,
))

fig.update_layout(
    title={
        "text": "EGID Abdeckung –  Gebäude CH"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                f"Quelle: AV/GWR | Total: {total:,} Gebäude</span>"
    },
    uniformtext_minsize=14,
    uniformtext_mode="hide",
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="right", x=1.1)
)

fig.write_image("egid_coverage.png")
with open("egid_coverage.png.meta.json", "w") as f:
    json.dump({
        "caption": "EGID Abdeckung Gebäude Schweiz",
        "description": "Pie chart showing how many buildings have a GWR_EGID value vs missing"
    }, f)


['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
Total features:    3,349,331
Mit EGID:          3,349,331  (100.0%)
Ohne EGID:         0 (0.0%)


Anzahl Stockwerke

In [2]:
import geopandas as gpd
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file(
    "../whatamidoing.geojson"
)

print(gdf.columns.tolist())

# ── COUNT ─────────────────────────────────────────────────────────────────────
total = len(gdf)
has_val = gdf["gastw"].notna()
count_with    = has_val.sum()
count_without = (~has_val).sum()
pct_with    = count_with / total * 100
pct_without = count_without / total * 100

print(f"Total:          {total:,}")
print(f"Mit Wert:       {count_with:,}  ({pct_with:.1f}%)")
print(f"Keine Angabe:   {count_without:,} ({pct_without:.1f}%)")

# ── PIE CHART ─────────────────────────────────────────────────────────────────
fig = go.Figure(go.Pie(
    labels=["Mit Wert", "Keine Angabe"],
    values=[count_with, count_without],
    textinfo="label+percent",
    texttemplate="%{label}<br>%{percent}<br>%{value:,}",
    hole=0.3,
))

fig.update_layout(
    title={
        "text": "Anzahl Geschosse – Datenvollständigkeit CH"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                f"Quelle: AV/GWR | Total: {total:,} Gebäude</span>"
    },
    uniformtext_minsize=14,
    uniformtext_mode="hide",
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="right", x=1.1)
)

fig.write_image("geschosse_coverage.png")
with open("geschosse_coverage.png.meta.json", "w") as f:
    json.dump({
        "caption": "Datenvollständigkeit Anzahl Geschosse",
        "description": "Pie chart showing share of buildings with and without a floor count value"
    }, f)


['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
Total:          3,349,331
Mit Wert:       2,090,125  (62.4%)
Keine Angabe:   1,259,206 (37.6%)


has EGID Value by Canton


In [24]:
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file(
    "../../Geobasiszwilling/geobasiszwilling_fhnw/import/proj/projected_buildings_enriched_ch.geojson"
)

print(gdf.columns.tolist())
gdf["GWR_EGID"] = pd.to_numeric(gdf["GWR_EGID"], errors="coerce")

# ── COUNT PER CANTON ──────────────────────────────────────────────────────────
canton_stats = gdf.groupby("Kanton").apply(
    lambda x: pd.Series({
        "Mit EGID":    (x["GWR_EGID"].notna() & (x["GWR_EGID"] != 0)).sum(),
        "Ohne EGID":   (x["GWR_EGID"].isna()  | (x["GWR_EGID"] == 0)).sum(),
        "Total":       len(x),
    })
).reset_index()

canton_stats["% Mit EGID"] = (canton_stats["Mit EGID"] / canton_stats["Total"] * 100).round(1)
canton_stats = canton_stats.sort_values("Total", ascending=False)

print(canton_stats.to_string(index=False))

# ── GROUPED BAR CHART ─────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Mit EGID",
    x=canton_stats["Kanton"],
    y=canton_stats["Mit EGID"],
    # text=canton_stats["Mit EGID"].apply(lambda v: f"{v:,}"),
    textposition="outside",
))

fig.add_trace(go.Bar(
    name="Ohne EGID",
    x=canton_stats["Kanton"],
    y=canton_stats["Ohne EGID"],
    # text=canton_stats["Ohne EGID"].apply(lambda v: f"{v:,}"),
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    barmode="group",
    title={
        "text": "EGID Abdeckung pro Kanton"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: AV/GWR </span>"
    },
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Kanton")
fig.update_yaxes(title_text="Anzahl Gebäude", range=[0, 6000])

fig.write_image("egid_coverage_kanton.png")
with open("egid_coverage_kanton.png.meta.json", "w") as f:
    json.dump({
        "caption": "EGID Abdeckung pro Kanton (Mit vs. Ohne)",
        "description": "Grouped bar chart showing buildings with and without EGID per Swiss canton, sorted by coverage rate"
    }, f)

# ── ALSO SAVE STATS AS CSV ────────────────────────────────────────────────────
canton_stats.to_csv("egid_coverage_kanton.csv", index=False)
print("\nSaved → egid_coverage_kanton.png + egid_coverage_kanton.csv")


['BFSNr', 'Qualitaet', 'Art', 'GWR_EGID', 'Kanton', 'gastw', 'Adresse', 'Gemeinde', 'Parzellennummer', 'gstat', 'gstat_text', 'gkat', 'gkat_text', 'garea', 'gvol', 'gebf', 'gvolsce', 'gvolsce_text', 'geometry']
Kanton  Mit EGID  Ohne EGID  Total  % Mit EGID
    ZH      4502       2081   6583        68.4
    GE      5798          0   5798       100.0
    AG      4378       1358   5736        76.3
    SG      2776       1919   4695        59.1
    FR      2729       1688   4417        61.8
    BE      3650        458   4108        88.9
    TI      1897        784   2681        70.8
    TG       650       1841   2491        26.1
    GR      1429        919   2348        60.9
    BL      1509         24   1533        98.4
    SZ      1138        243   1381        82.4
    SO       491        531   1022        48.0
    ZG       447        330    777        57.5
    SH         0        656    656         0.0
    VS         0        519    519         0.0
    AR       258        245    503   

stockwerk value by kanton

In [ ]:
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file(
    "../whatamidoing.geojson"
)

print(gdf.columns.tolist())
gdf["gastw"] = pd.to_numeric(gdf["gastw"], errors="coerce")

# ── COUNT PER CANTON ──────────────────────────────────────────────────────────
canton_stats = gdf.groupby("canton").apply(
    lambda x: pd.Series({
        "Mit Stockwerk":    (x["gastw"].notna() & (x["gastw"] != 0)).sum(),
        "Ohne Stockwerk":   (x["gastw"].isna()  | (x["gastw"] == 0)).sum(),
        "Total":       len(x),
    })
).reset_index()

canton_stats["% Mit Stockwerk"] = (canton_stats["Mit Stockwerk"] / canton_stats["Total"] * 100).round(1)
canton_stats = canton_stats.sort_values("Total", ascending=False)

print(canton_stats.to_string(index=False))

# ── GROUPED BAR CHART ─────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Mit Stockwerk",
    x=canton_stats["canton"],
    y=canton_stats["Mit Stockwerk"],
    # text=canton_stats["Mit Stockwerk"].apply(lambda v: f"{v:,}"),
    textposition="outside",
))

fig.add_trace(go.Bar(
    name="Ohne Stockwerk",
    x=canton_stats["canton"],
    y=canton_stats["Ohne Stockwerk"],
    # text=canton_stats["Ohne Stockwerk"].apply(lambda v: f"{v:,}"),
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    barmode="group",
    title={
        "text": "Stockwerk Abdeckung pro Kanton"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: AV/GWR </span>"
    },
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Kanton")
fig.update_yaxes(title_text="Anzahl Gebäude")

fig.write_image("stockwerk_coverage_kanton.png")
with open("stockwerk_coverage_kanton.png.meta.json", "w") as f:
    json.dump({
        "caption": "Stockwerk Abdeckung pro Kanton (Mit vs. Ohne)",
        "description": "Grouped bar chart showing buildings with and without Stockwerk per Swiss canton, sorted by coverage rate"
    }, f)

# ── ALSO SAVE STATS AS CSV ────────────────────────────────────────────────────
canton_stats.to_csv("stockwerk_coverage_kanton.csv", index=False)
print("\nSaved → stockwerk_coverage_kanton.png + stockwerk_coverage_kanton.csv")


['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
canton  Mit Stockwerk  Ohne Stockwerk  Total  % Mit Stockwerk
    BE         259565          217125 476690             54.5
    ZH         263748          135368 399116             66.1
    AG         169884          117336 287220             59.1
    VD         153258           98675 251933             60.8
    SG         125879           97881 223760             56.3
    VS         1348

KeyError: 'Kanton'

stockwerk value by kanton prozentual

In [5]:
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file(
    "../whatamidoing.geojson"
)

print(gdf.columns.tolist())
gdf["gastw"] = pd.to_numeric(gdf["gastw"], errors="coerce")

# ── COUNT PER CANTON ──────────────────────────────────────────────────────────
canton_stats["% Mit Stockwerk"] = (
    canton_stats["Mit Stockwerk"] / canton_stats["Total"] * 100
).round(1)
canton_stats = canton_stats.sort_values("% Mit Stockwerk", ascending=False)

print(canton_stats.to_string(index=False))

# ── SINGLE BAR CHART: PERCENT COVERAGE ────────────────────────────────────────
fig = go.Figure(go.Bar(
    name="% Mit Stockwerk",
    x=canton_stats["canton"],
    y=canton_stats["% Mit Stockwerk"],
    text=canton_stats["% Mit Stockwerk"].apply(lambda v: f"{v:.1f}%"),
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    title={
        "text": "Stockwerk-Abdeckung pro Kanton (% mit Wert)"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: AV/GWR</span>"
    },
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Kanton")
fig.update_yaxes(title_text="Abdeckung [%]", range=[0, 100])

fig.write_image("stockwerk_coverage_kanton_pct.png")
with open("stockwerk_coverage_kanton_pct.png.meta.json", "w") as f:
    json.dump({
        "caption": "Stockwerk-Abdeckung pro Kanton in Prozent",
        "description": "Bar chart showing percentage of buildings with non-zero gastw per canton"
    }, f)

canton_stats.to_csv("stockwerk_coverage_kanton_pct.csv", index=False)
print("\nSaved → stockwerk_coverage_kanton_pct.png + stockwerk_coverage_kanton_pct.csv")


['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
canton  Mit Stockwerk  Ohne Stockwerk  Total  % Mit Stockwerk
    BS          30463            1784  32247             94.5
    TI         183270           13565 196835             93.1
    BL          97561           31833 129394             75.4
    GE          62044           25477  87521             70.9
    ZH         263748          135368 399116             66.1
    SO          757

hat Volumen und Fläche

In [6]:
import geopandas as gpd
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file(
    "../whatamidoing.geojson"
)

print(gdf.columns.tolist())

# ── COUNT ─────────────────────────────────────────────────────────────────────
total = len(gdf)
has_val = gdf["garea"].notna() & gdf["gvol"].notna()
count_with    = has_val.sum()
count_without = (~has_val).sum()
pct_with    = count_with / total * 100
pct_without = count_without / total * 100

print(f"Total:          {total:,}")
print(f"Mit Wert:       {count_with:,}  ({pct_with:.1f}%)")
print(f"Keine Angabe:   {count_without:,} ({pct_without:.1f}%)")

# ── PIE CHART ─────────────────────────────────────────────────────────────────
fig = go.Figure(go.Pie(
    labels=["Mit Wert", "Keine Angabe"],
    values=[count_with, count_without],
    textinfo="label+percent",
    texttemplate="%{label}<br>%{percent}<br>%{value:,}",
    hole=0.3,
))

fig.update_layout(
    title={
        "text": "Fläche und Volumen – Datenvollständigkeit CH"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                f"Quelle: AV/GWR | Total: {total:,} Gebäude</span>"
    },
    uniformtext_minsize=14,
    uniformtext_mode="hide",
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="right", x=1.1)
)

fig.write_image("vol_coverage.png")
with open("vol_coverage.png.meta.json", "w") as f:
    json.dump({
        "caption": "Datenvollständigkeit Fläche und Volumen",
        "description": "Pie chart showing share of projected buildings with and without area and volume values"
    }, f)


['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
Total:          3,349,331
Mit Wert:       276,507  (8.3%)
Keine Angabe:   3,072,824 (91.7%)


Volumen und Flächen pro Kanton

In [ ]:
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
import json

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
gdf = gpd.read_file("../whatamidoing.geojson")
print(gdf.columns.tolist())

gdf["garea"] = pd.to_numeric(gdf["garea"], errors="coerce")
gdf["gvol"]  = pd.to_numeric(gdf["gvol"],  errors="coerce")

# ── COUNT PER CANTON ──────────────────────────────────────────────────────────
canton_stats = gdf.groupby("canton").apply(
    lambda x: pd.Series({
        "Mit Fläche und Volumen":  (x["garea"].notna() & x["gvol"].notna()).sum(),
        "Ohne Fläche und Volumen": (x["garea"].isna()  | x["gvol"].isna()).sum(),
        "Total":                   len(x),
    })
).reset_index()

canton_stats["% Mit Fläche und Volumen"] = (
    canton_stats["Mit Fläche und Volumen"] / canton_stats["Total"] * 100
).round(1)

# Sort by percentage coverage
canton_stats = canton_stats.sort_values("% Mit Fläche und Volumen", ascending=False)

print(canton_stats.to_string(index=False))

# ── BAR CHART: PERCENT COVERAGE ───────────────────────────────────────────────
fig = go.Figure(go.Bar(
    name="% Mit Fläche und Volumen",
    x=canton_stats["canton"],
    y=canton_stats["% Mit Fläche und Volumen"],
    text=canton_stats["% Mit Fläche und Volumen"].apply(lambda v: f"{v:.1f}%"),
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    title={
        "text": "Fläche & Volumen – Abdeckung pro Kanton (%)"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: AV/GWR</span>"
    },
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5)
)
fig.update_xaxes(title_text="Kanton")
fig.update_yaxes(title_text="Abdeckung [%]", range=[0, 100])

fig.write_image("vol_coverage_kanton_pct.png")
with open("vol_coverage_kanton_pct.png.meta.json", "w") as f:
    json.dump({
        "caption": "Fläche & Volumen – Abdeckung pro Kanton in Prozent",
        "description": "Bar chart showing percentage of buildings with both garea and gvol per Swiss canton"
    }, f)

canton_stats.to_csv("vol_coverage_kanton_pct.csv", index=False)
print("\nSaved → vol_coverage_kanton_pct.png + vol_coverage_kanton_pct.csv")

['BFSNr', 'Qualitaet', 'Art', 'GWR_EGID', 'Kanton', 'gastw', 'Adresse', 'Gemeinde', 'Parzellennummer', 'gstat', 'gstat_text', 'gkat', 'gkat_text', 'garea', 'gvol', 'gebf', 'gvolsce', 'gvolsce_text', 'geometry']
Kanton  Mit Fläche und Volumen  Ohne Fläche und Volumen  Total  % Mit Fläche und Volumen
    ZH                    3601                     2982   6583                      54.7
    GE                       0                     5798   5798                       0.0
    AG                    3592                     2144   5736                      62.6
    SG                    2253                     2442   4695                      48.0
    FR                    2051                     2366   4417                      46.4
    BE                    3008                     1100   4108                      73.2
    TI                    1353                     1328   2681                      50.5
    TG                     328                     2163   2491               

Anzahl Geschosse Alle Gebäude

In [3]:
import geopandas as gpd
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import json

# ── LOAD DATA FROM GEOJSON ────────────────────────────────────────────────────
gdf = gpd.read_file(
    "whatamidoing_test.geojson"
)

print(gdf.columns.tolist())


# ── KEY FIX: fill NaN BEFORE value_counts ────────────────────────────────────
col = "gastw"   # change if the name differs in the GeoJSON
series = gdf[col].fillna("Keine Angabe").astype(str)

counts = series.value_counts().reset_index()
counts.columns = ["Geschosse", "Anzahl"]

def sort_key(val):
    try:
        return (0, float(val))
    except:
        return (1, val)

counts = counts.sort_values("Geschosse", key=lambda x: x.map(sort_key)).reset_index(drop=True)

# ── CHART ─────────────────────────────────────────────────────────────────────
fig = go.Figure(go.Bar(
    x=counts["Geschosse"],
    y=counts["Anzahl"],
    textposition="outside",
))

fig.update_traces(cliponaxis=False)
fig.update_layout(
    title={
        "text": "Anzahl Geschosse – Verteilung CH (GeoJSON)"
                "<br><span style='font-size:18px;font-weight:normal;'>"
                "Quelle: projected_buildings_enriched_ch.geojson | inkl. fehlende Werte</span>"
    }
)
fig.update_xaxes(title_text="Geschosse", type="category")
fig.update_yaxes(title_text="Anzahl Gebäude")

fig.write_image("anzahl_geschosse_alle_geojson.png")
with open("anzahl_geschosse_alle_geojson.png.meta.json", "w") as f:
    json.dump({
        "caption": "Verteilung Anzahl Geschosse (GeoJSON, inkl. fehlende Werte)",
        "description": "Bar chart of floor count distribution from projected_buildings_enriched_ch.geojson including missing values"
    }, f)

print(counts.to_string(index=False))

['egid', 'buildingStatus', 'buildingCategory', 'buildingClass', 'municipalityNumber', 'municipalityName', 'canton', 'energyReferenceArea', 'heating1_heatGenerator', 'heating1_energySource', 'heating1_informationSource', 'heating1_revisionDate', 'heating2_heatGenerator', 'heating2_energySource', 'heating2_informationSource', 'heating2_revisionDate', 'hotwater1_heatGenerator', 'hotwater1_energySource', 'hotwater1_informationSource', 'hotwater1_revisionDate', 'hotwater2_heatGenerator', 'hotwater2_energySource', 'hotwater2_informationSource', 'hotwater2_revisionDate', 'garea', 'gvol', 'gastw', 'geometry']
   Geschosse  Anzahl
         1.0      19
         2.0     220
         3.0     168
         4.0      45
         5.0      10
         6.0       3
         7.0       1
         8.0       2
Keine Angabe 3348863
